# Imports

In [1]:
from utils.dropbox_filesystem import get_all_paths, sort_source_to_target
from utils.save_logs import save_file_infos, save_jsons_to_data, correct_file_infos_with_matching_metadata 
from utils.save_logs import write_paths_file, write_general_recap_file, merge_general_recaps, refresh_new_paths
from utils.handle_duplicates import flag_potential_duplicates, rename_duplicates

from tokens import ACCESS_TOKEN


# Get the paths

In [2]:
input_files = get_all_paths(TOKEN= ACCESS_TOKEN, 
                            dir= '/source', 
                            recursive=True, 
                            remove_source=True
                            )
print('Done \n')

#for path in input_files:
#    print(path)

Creating a Dropbox object...
Access token is valid
Done 



In [3]:
for path in input_files:
    print(path)

/TG2001_Brain_NIFTI/Structural/mp2rage/TG2001_Brain_wip19_mp2rage_1iso_cs4p6_20240515151828_12.nii.gz
/TG2001_Brain_NIFTI/Structural/mp2rage/TG2001_Brain_wip19_mp2rage_1iso_cs4p6_20240515151828_6.nii.gz
/TG2001_Brain_NIFTI/Structural/mp2rage/TG2001_Brain_wip19_mp2rage_1iso_cs4p6_20240515151828_8.nii.gz
/TG2001_Brain_NIFTI/Structural/mp2rage/TG2001_Brain_wip19_mp2rage_1iso_cs4p6_20240515151828_10.nii.gz
/TG2001_Brain_NIFTI/Structural/mp2rage/TG2001_Brain_wip19_mp2rage_1iso_cs4p6_20240515151828_11.nii.gz
/TG2001_Brain_NIFTI/Structural/mp2rage/TG2001_Brain_wip19_mp2rage_1iso_cs4p6_20240515151828_7.nii.gz
/TG2001_Brain_NIFTI/Structural/TG2001_Brain_wip19_mp2rage_1iso_cs4p6_20240515151828_12.nii.gz
/TG2001_Brain_NIFTI/Structural/TG2001_Brain_t1_spc3d_sag_20240515151828_5.nii.gz
/TG2001_Brain_NIFTI/Structural_Tissues/TG2001_Brain_t1_spc3d_sag_20240515151828_5_seg_post_pro_right_wm.nii.gz
/TG2001_Brain_NIFTI/Structural_Tissues/TG2001_Brain_wip19_mp2rage_1iso_cs4p6_20240515151828_12_seg_post_p

# Prepare all paths for the pipeline

In [ ]:
subdir = 't2g'

n = '05'

file_infos_path = 'file_infos/'+ subdir +'/file_infos-' + n + '.json'

jsons_to_data_path = 'file_infos/'+ subdir +'/jsons_to_data-' + n + '.json'

corrected_file_infos_path = file_infos_path[:-len('.json')] +'_corrected.json'

flagged_path = 'file_infos/'+ subdir +'/potential_duplicates-' + n + '.json'

renamed_duplicates_file_infos_path = corrected_file_infos_path[:-len('.json')] +'_renamed_duplicates.json'


txt_logs_path = 'paths/' + subdir + '/paths-' + n + '.txt'

json_recap_path = 'recaps/' + subdir + '/recap-' + n + '.json'


old_prefix = 'TG2001/Brain_Imaging'

# Save file infos

In [ ]:
#save_file_infos(
#    input_files,
#    participants_dict={
#        "old_sub": "sub-[n]"
#    },
#    out_path='file_infos/blabla.json',
#    **kwargs
#    )



save_file_infos(
    input_files,
    participants_dict={
        "TG2001": "sub-01",
        "T2G001": "sub-01",
        "T2G002": "sub-02",
        "T2G003": "sub-03",
        "T2G004": "sub-04"
    },
    out_path= file_infos_path,
    sub = 'sub-01')

done


If the file infos were manually modified, it is possible to refresh the new paths:

In [3]:
refresh_new_paths(
    file_infos_path= renamed_duplicates_file_infos_path,
    new_file_infos_path = renamed_duplicates_file_infos_path
)

## Handling duplicates

In [ ]:
save_jsons_to_data(
    file_infos_path = file_infos_path,
    jsons_to_data_path= jsons_to_data_path
)

**TO DO:**
> Manually correct the json in `jsons_to_data_path` to match each json to its correct data file (usually nifti).

Correct file infos:

In [6]:
correct_file_infos_with_matching_metadata(
    file_infos_path = file_infos_path,
    jsons_to_data_path = jsons_to_data_path,
    corrected_file_infos_path = corrected_file_infos_path
)

Handle possible duplicates

In [7]:
flag_potential_duplicates(
    file_infos_path=corrected_file_infos_path,
    flagged_path= flagged_path
)

In [8]:
rename_duplicates(
    corrected_file_infos_path,
    flagged_path,
    renamed_duplicates_file_infos_path
)

In [4]:
# Second duplicate check
flag_potential_duplicates(
    file_infos_path=renamed_duplicates_file_infos_path,
    flagged_path= flagged_path
)



# Save `paths.txt` and `recap.json`

In [5]:


write_paths_file(
    file_infos_path= renamed_duplicates_file_infos_path,
    out_path = txt_logs_path,
    old_prefix=old_prefix,
    new_prefix=''
    )

In [6]:
write_general_recap_file(
    file_infos_path= renamed_duplicates_file_infos_path,
    out_path= json_recap_path,
    new_prefix=''
    )

Eventually merge several json recaps:

In [ ]:
merge = False

if merge:

    merge_general_recaps(
        input_files=[
            'recaps/recap_01.json',
            'recaps/recap_02.json'
        ],
        out_path='recaps/merged_recap_test.json'
    )

# Copy and rename old files to their new paths

In [4]:
sort_source_to_target(
    file_infos_path= renamed_duplicates_file_infos_path,
    TOKEN=ACCESS_TOKEN
    )

Creating a Dropbox object...
Access token is valid


100%|██████████| 348/348 [07:51<00:00,  1.36s/it]
